# Group 42

Main

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import time
import asyncio

from thymio import Thymio

import vision
import motion_controll
import filtering
import local_nav
import global_nav

thymio = Thymio(pos_init=[0, 0], orient=0)
await thymio._connect_to_thymio_()
thymio.stop()

Local nav tests

In [ ]:
thymio.nav_mode = "LOCAL"
while True: 
    await thymio.update_ir()
    # is_object = local_nav.is_object(thymio)
    # if(is_object):
    #     print("object detected")
    print(str(thymio.ir_sensors) + "                     ", end="\r")
    # avoid_right = local_nav.avoid_right(thymio, grid)
    # local_nav.avoid_obstacle(thymio, 0, 0, avoid_right)
# await thymio.update_ir()
# print(thymio.ir_sensors)

Motion controll testing

In [ ]:
thymio.pos = [20,20]
thymio.orient = 0
goal = [22,25]
print(motion_controll.follow_path(thymio, goal))

In [ ]:
await thymio.unlock()

## Main Loop

In [ ]:
thymio.stop()

In [ ]:
# External modules import
import matplotlib.pyplot as plt
import numpy as np
import cv2
import time
import asyncio

# Internal modules and thymio class import
from thymio import Thymio
from vision import Vision
import motion_controll
import filtering
import local_nav
import global_nav
import utils

# Constants
GL_NAV_CHANGE_THLD = 20

# Initalisation of the grid
visionInstance = Vision() # Calls getEnvironment which stores the arena and creates the grid
visionInstance.display_grid()

# Get cell size for coordinate conversions
cell_size_cm = visionInstance.getCellSizeCm()

# Get start position and orientation
ret, frame = visionInstance.cap.read() #Taking a single image to find the start pos of the robot
if not ret:
    raise ValueError("Camera failed to capture the frame.")

#start_pos, start_orient = visionInstance.getRobotPose(frame)
start_pos, start_orient = visionInstance.getInitialRobotPose()
print(start_pos, start_orient)

# Convert to grid coordinates (start_pos is in meters, convert to cm first)
if start_pos:
    start_cell = utils.real_to_grid((start_pos[0]*100, start_pos[1]*100))
    print(f"start: {start_cell}")
    goal_cell = utils.real_to_grid((5, 5))  # Goal in cm
    print(f"goal: {goal_cell}")
    # visionInstance.display_grid(start_cell, goal_cell)
    # Find the path to goal
    #path = global_nav.find_path(visionInstance.grid, start_cell, goal_cell)

# Get the grid from vision
grid = visionInstance.grid

# ----------------------RUN EXPANDED MAP AND DIJKSTRA -----------------------
path, expanded_grid = global_nav.expanded_dijkstra(grid, start_cell, goal_cell)

if path:
    simplified_path = global_nav.simplify_path(path)
    global_nav.display_map(expanded_grid, path, simplified_path, start_cell, goal_cell)

    print("--- Dijkstra Pathfinding Results ---")
    print(f"Simplified path : {simplified_path}")
    converted_simplified_path = [utils.grid_to_real(p) for p in simplified_path]# Convert to pure Python floats in lists
    real_waypts = [[float(wp[0]), float(wp[1])] for wp in converted_simplified_path]
    print("Simplified real Waypoints (real cm):", real_waypts)
    print("------------------------------")
else:
      print("No path found.")  
# ---------------------- END OF EXPANDED MAP AND DIJKSTRA ---------------------

#---------------------- RUN A* ---------------------
# path = global_nav.find_path(grid, start_cell, goal_cell)
# if path:
#     simplified_path = global_nav.simplify_path(path)
    
#     print("--- A* Pathfinding Results ---")
#     # print(f"Full Path Length (cells): {len(path)-1}")
#     print(f"Simplified Path Waypoints: {len(simplified_path)}")
#     print(simplified_path)
#     converted_simplified_path = [utils.grid_to_real(p) for p in simplified_path]
#     # Convert to pure Python floats in lists
#     real_waypts = [[float(wp[0]), float(wp[1])] for wp in converted_simplified_path]
#     print("Simplified real Waypoints (real cm):", real_waypts)
#     print("------------------------------")
#     global_nav.display_map(visionInstance.grid, path, simplified_path, start_cell, goal_cell)
# else:
#     print("No path found.")
#---------------------- END A* ---------------------



# Initialisation of the thymio and connexion
thymio = Thymio(pos_init=start_pos, orient=start_orient)
await thymio._connect_to_thymio_()

thymio.stop()


gl_nav_change_cntr = 0

while(True):
    thymio.update_ir()
    # print(thymio.ir_sensors)
    is_object = local_nav.is_object(thymio)
    nav_mode_change = False
    
    if(not is_object and thymio.nav_mode=="LOCAL"):
        gl_nav_change_cntr += 1
        if(gl_nav_change_cntr >= GL_NAV_CHANGE_THLD):
            nav_mode_change = True
            thymio.nav_mode = "GLOBAL"
            path = global_nav.find_path()

    if(is_object):
        gl_nav_change_cntr = 0
        if(thymio.nav_mode=="GLOBAL"):
            nav_mode_change = True
            thymio.nav_mode = "LOCAL"
        local_nav.avoid_obstacle(thymio, grid=grid, path=path)
    
    if(thymio.nav_mode=="GLOBAL"):
        next_wp = real_waypts[0]
        # next_wp = [35.0, 35.0]  # Copy to avoid modifying the original
        # print(f"thymio pos: {thymio.pos}, next wp: {next_wp}", end="\r")
        wp_reached = motion_controll.follow_path(thymio, next_wp)
        if(wp_reached):
            print("Waypoint reached!")
            real_waypts.pop(0)  # Supprime le waypoint atteint
            if len(real_waypts) == 0:  # Si plus de waypoints
                print("Destination atteinte!")
                thymio.stop()
                break  # Sortir de la boucle
    
    # pos_on_img, orient_on_img = vision.get_pos()
    # Get robot position from vision 
    ret, frame = visionInstance.cap.read() 
    if not ret:
        print("Camera failed to capture the frame.")
        break

    vis = frame.copy() # Copying frame so we can display shapes on top without affecting detection
    visionInstance.visualiseArena(vis, visionInstance.arena_corners_pixels)

    # --- Always detect and draw robot pose ---
    result = visionInstance.getRobotPoseAndVisualise(frame, vis)
    
    # Check if detection failed (returns (None, None))
    if result[0] is None:
        for try_get_thymio in range(5):
            ret, frame = visionInstance.cap.read() 
            if not ret:
                print("Camera failed to capture the frame.")
                break
            vis = frame.copy() # Copying frame so we can display shapes on top without affecting detection
            visionInstance.visualiseArena(vis, visionInstance.arena_corners_pixels)
            result = visionInstance.getRobotPoseAndVisualise(frame, vis)
            if result[0] is not None:
                break

    if result[0] is None:
        thymio.stop()
        print("Robot detection failed after multiple attempts.")
        break
    
    [X_robot, Y_robot], robot_heading_angle = result

    print(f"X: {X_robot:.5f}, Y: {Y_robot:.5f}, Direction: {robot_heading_angle:.5f}", end="\r")

    thymio.pos = [X_robot*100, Y_robot*100]  # Convert to cm
    thymio.orient = robot_heading_angle


    # filtering.filter_pos(thymio, pos_on_img, orient_on_img)

In [ ]:
thymio.stop()

In [ ]:

# Get robot pose in loop
if not visionInstance.cap.isOpened(): # Checking access to camera feed
    print("Error: Could not access the webcam.")
    exit()

while True:
    ret, frame = visionInstance.cap.read() # Taking an image frame from the camera feed
    if not ret:
        print("Camera failed to capture the frame.")
        break

    vis = frame.copy() # Copying frame so we can display shapes on top without affecting detection

    # --- Always redraw arena outline ---
    visionInstance.visualiseArena(vis, visionInstance.arena_corners_pixels)

    # --- Always detect and draw robot pose ---
    [X, Y], robot_heading_angle = visionInstance.getRobotPoseAndVisualise(frame, vis)
    if (X is None or Y is None or robot_heading_angle is None):
        continue
    print(f"X: {X:.5f}, Y: {Y:.5f}, Direction: {robot_heading_angle:.5f}")

    # --- Show the live window ---
    cv2.imshow("Live Robot Pose", vis)

    # Exit on Q
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

visionInstance.cap.release()
cv2.destroyAllWindows()

In [ ]:
a = [[1,2],[3,4]]
a[1][1]

In [ ]:
await thymio.update_ir()
print(thymio.ir_sensors[0:5])

In [ ]:
thymio.set_motor_speeds([100, 100])
while True:
    await thymio.update_ir()
    if sum(thymio.ir_sensors) > 2000:
        thymio.stop()
        break
    

Next cell makes the Thymio robot move forward for 4 seconds and then stops each time the Forward button is pressed. Program stops when the Backward button is pressed.  
It is intended to collect data for computing the **velocity variance**.

In [ ]:
await thymio.button_loop()

Using this program to measure (with a ruler) the distance travelled by the bot each time to see differencies despite constant time and compute the variance on speed state.

In [ ]:
data_velocity_error=[160, 161, 159] #distances in mm travelled at presumed same speed for a constant time
data_velocity_error/=4 #distances divided by the constant time to get true velocities

q_v = np.var(data_velocity_error) # variance on speed state

From the camera we get a data set of XY position measurements from the same position to search for some differencies and compute variances on XY states and measurements.

In [ ]:
measurements_from_camera=np.array[[]]
x_measurements = [x[0] for x in measurements_from_camera]
y_measurements = [y[1] for y in measurements_from_camera]

var_x = np.var(x_measurements)
var_y = np.var(y_measurements)

q_x = var_x/2 #variance on x position state
r_x = var_x/2 #variance on x position measurement
q_y = var_y/2 #variance on y position state
r_y = var_y/2 #variance on y position measurement

In [ ]:
def get_vector_from_circles(image):

    detector = cv2.SimpleBlobDetector_create()
    circle_centers = detector.detect(image)
    p1 = np.array(circle_centers[0].pt)
    p2 = np.array(circle_centers[1].pt)
    return p2-p1

In [ ]:
def test_get_vector_from_circles(image):
    return [0,0]

In [ ]:
thymio.set_motor_speeds([0,0])
# ToDo: start camera
image = 0 #ToDo
delta_t = 2
start_time=0
k=1
data_vectors_pos=np.zeros((k, 2, 2))
data_vectors_neg=np.zeros((k, 2, 2))
for i in range(k):

    data_vectors_pos[i][0]=test_get_vector_from_circles(image)
    thymio.set_motor_speeds([100,-100])
    start_time=time.time()
    while True:
        if time.time() - start_time >=delta_t:
            thymio.set_motor_speeds([0,0])
            break
    data_vectors_pos[i][1]=test_get_vector_from_circles(image)
    data_vectors_neg[i][0]=test_get_vector_from_circles(image)

    thymio.set_motor_speeds([-100,100])
    start_time=time.time()
    while True:
        if time.time() - start_time >=delta_t:
            thymio.set_motor_speeds([0,0])
            break
    data_vectors_neg[i][1]=test_get_vector_from_circles(image)
    
    